# AAROH Mental Health Language Representation

Train a representation-only multilingual language encoder on MindBridge. This model does not predict clinical scores or diagnoses.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
REPO_DIR = '/content/AAROH'
if not os.path.isdir(REPO_DIR): raise FileNotFoundError(f'Expected repository at {REPO_DIR}')
os.chdir(REPO_DIR)

In [ ]:
%pip install -q torch transformers scikit-learn
import torch
if not torch.cuda.is_available(): raise RuntimeError('A CUDA runtime is required for transformer training')
print('CUDA:', torch.cuda.get_device_name(0))

In [ ]:
from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive/AAROH')
CHECKPOINT_DIR = DRIVE_ROOT / 'checkpoints' / 'mental_health_language'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
Path('models/mental_health_language').mkdir(parents=True, exist_ok=True)

In [ ]:
!python -m backend.ml.training.train_mental_health \
  --data-dir datasets/processed \
  --output-dir models/mental_health_language \
  --checkpoint-dir checkpoints/mental_health_language \
  --drive-checkpoint-dir $CHECKPOINT_DIR \
  --model-name distilbert-base-uncased \
  --execution-mode PYTORCH_FROZEN \
  --epochs 5 \
  --batch-size 16 \
  --fp16

In [ ]:
from backend.ml.training.models.mental_health_language.model import MentalHealthLanguageModel
model = MentalHealthLanguageModel.load_from_artifact('models/mental_health_language', device='cuda')
result = model.encode(['I have been reflecting on my wellbeing.'], device='cuda')
assert len(result['mental_health_embeddings']) == 1
assert len(result['mental_health_embeddings'][0]) == 768
print('Representation inference verification passed')

In [ ]:
from pathlib import Path
required = ['config.json','metadata.json','pytorch_model.bin','tokenizer.json','tokenizer_config.json','label_mapping.json','metrics.json']
missing = [name for name in required if not (Path('models/mental_health_language') / name).exists()]
if missing: raise FileNotFoundError(missing)
print('Mental Health artifacts exported:', required)